In [2]:
import batman
import emcee
import glob
import os
import shutil
import math
import corner
import numba
import itertools
import sys

import numpy       as np
import pandas      as pd
import time        as tm 
import lightkurve  as lk
# import mr_forecast as mr

import matplotlib                      as mpl
import matplotlib.pyplot               as plt
import matplotlib.gridspec             as gridspec
from   matplotlib.backends.backend_pdf import PdfPages
import mpl_axes_aligner


import astropy.io.fits    as apf
import astropy.units      as units
from   astropy.stats      import sigma_clip
from   astropy.wcs        import WCS
from   astroquery.mast    import Catalogs
from   astroquery         import svo_fps

from multiprocessing import Pool, Process
from wotan           import flatten
from functools       import partial
from ldtk            import LDPSetCreator, BoxcarFilter, TabulatedFilter, SVOFilter
from ldtk.filters    import tess, sdss_z
from IPython.display import display, HTML
from tqdm.auto       import tqdm


# import eleanor

# import warnings
# warnings.filterwarnings("ignore")
# display(HTML("<style>.container { width:95% !important; }</style>"))
module_path = os.path.abspath(os.path.join('..', 'src'))
sys.path.insert(0, module_path)
from dr2_to_dr3_conversion_second_attempt import *
import gc



In [3]:
from astroquery.gaia import Gaia


The archive is unstable and may perform below expectations. If launching multiple, consecutive, heavy queries through Python, please space them out (e.g., using sleep(1)) to avoid overloading the system. Please contact the Gaia helpdesk in case of questions (https://www.cosmos.esa.int/web/gaia/gaia-helpdesk). Workaround solutions for the issues following the December 2025 infrastructure upgrade: https://www.cosmos.esa.int/web/gaia/news#WorkaroundArchive


In [4]:
df = pd.read_parquet('../data/dr2_to_dr3_results/TIC_and_Gaia_Dr3_ids_sector_07_dr2_to_dr3_chunk_0002.parquet')
df.head()

,dr2_source_id,dr3_source_id,angular_distance,magnitude_difference
0,2894329154550379392,2894329158850717440,0.433113,-0.028021
1,2894330292720917760,2894330292720917760,0.075032,-0.032036
2,2894331460952000768,2894331460952000768,0.049759,-0.021804
3,2894335751620348416,2894335751620348416,0.066911,-0.025524
4,2894353584326475392,2894353584326475392,0.192848,-0.027239


In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import pandas as pd
from pathlib import Path

def consolidate_results(
    parquet_dir: str,
    csv_dir: str,
    out_csv: str = "final_dr2_dr3_with_ticid.csv",
    out_parquet: str = "final_dr2_dr3_with_ticid.parquet"
):
    parquet_dir = Path(parquet_dir)
    csv_dir     = Path(csv_dir)

    # --- 1. Load all parquet files ---
    print("Loading parquet files...")
    pq_files = sorted(parquet_dir.glob("*.parquet"))
    if not pq_files:
        raise RuntimeError(f"No parquet files found in {parquet_dir}")
    
    pq_frames = []
    for pq in pq_files:
        df = pd.read_parquet(pq)
        # Ensure correct dtypes
        for c in ["dr2_source_id", "dr3_source_id"]:
            if c in df.columns:
                df[c] = df[c].astype("int64")
        pq_frames.append(df)

    df_parquet = pd.concat(pq_frames, ignore_index=True).drop_duplicates()
    print(f"Loaded {len(df_parquet):,} parquet rows (deduped).")

    # --- 2. Load all TIC/DR2 CSVs ---
    print("Loading TIC/DR2 source ID CSVs...")
    csv_files = sorted(csv_dir.glob("*.csv"))
    if not csv_files:
        raise RuntimeError(f"No CSV files found in {csv_dir}")
    
    tic_frames = []
    for cf in csv_files:
        df_csv = pd.read_csv(cf, names=["tic_id", "dr2_source_id"],
                             dtype={"tic_id":"int64","dr2_source_id":"int64"}, index_col = 0, header= 0)
        tic_frames.append(df_csv)

    df_ticmap = pd.concat(tic_frames, ignore_index=True).drop_duplicates()
    print(f"Loaded {len(df_ticmap):,} TIC/DR2 rows (deduped).")

    # --- 3. Merge parquet results with TIC IDs on dr2_source_id ---
    print("Merging TIC IDs with DR2→DR3 mappings...")

    df_merged = (
        df_parquet
        .merge(df_ticmap, on="dr2_source_id", how="left")
        .drop_duplicates()
    )

    # Reorder columns nicely
    cols_final = ["tic_id", "dr2_source_id", "dr3_source_id"]
    for extra in ["angular_distance", "score"]:
        if extra in df_merged.columns:
            cols_final.append(extra)
    df_merged = df_merged[cols_final]

    print(f"Merged final dataset: {len(df_merged):,} rows.")

    # --- 4. Write final outputs ---
    df_merged.to_csv(out_csv, index=False)
    df_merged.to_parquet(out_parquet, index=False)

    print(f"Written final CSV:    {out_csv}")
    print(f"Written final Parquet:{out_parquet}")

    # --- Optional diagnostics: unmatched TICIDs ---
    missing_tic = df_merged[df_merged["tic_id"].isna()]
    if len(missing_tic):
        warn_path = "unmatched_dr2_source_ids.csv"
        missing_tic.to_csv(warn_path, index=False)
        print(f"WARNING: {len(missing_tic):,} DR2 IDs had no TIC ID. Saved to {warn_path}")

    return df_merged

# from merge_ticids_and_dr3 import consolidate_results

consolidate_results(
    parquet_dir="../data/dr2_to_dr3_results/",
    csv_dir="../data/TIC_Gaia_Dr2_ids/",
    out_csv="../data/dr2_dr3_ticid_final.csv",
    out_parquet="../data/dr2_dr3_ticid_final.parquet"
)

Loading parquet files...
Loaded 2,178,408 parquet rows (deduped).
Loading TIC/DR2 source ID CSVs...
Loaded 2,178,408 TIC/DR2 rows (deduped).
Merging TIC IDs with DR2→DR3 mappings...
Merged final dataset: 2,178,408 rows.
Written final CSV:    ../data/dr2_dr3_ticid_final.csv
Written final Parquet:../data/dr2_dr3_ticid_final.parquet


,tic_id,dr2_source_id,dr3_source_id,angular_distance
0,141436378,4623777523893329664,4623777523893329664,0.062174
1,141436288,4623788549073928320,4623788549073928320,0.246775
2,141436282,4623788823951837696,4623788823951837696,0.146307
3,141436277,4623788995750531840,4623788995750531840,0.115750
4,141154297,4623903757276100352,4623903757276100352,0.037644
...,...,...,...,...
2178403,96344685,6031223940128059520,6031223940128059520,0.073912
2178404,95834645,6031246965475056256,6031246965475056256,0.128277
2178405,95834843,6031270158294411264,6031270158294411264,0.133901
2178406,95834986,6031271086018481536,6031271086018481536,0.143429


In [3]:
%%javascript
IPython.OutputArea.auto_scroll_threshold = 9999;



<IPython.core.display.Javascript object>

In [4]:
import pyvo

In [5]:


time1 = tm.time()
file_numbers = list(range(1, 14)) + list(range(27, 40))

for fi in file_numbers:
    # indx_num = 0
    file_name = '../data/TIC_and_Gaia_ids_sector_s' + str(fi).zfill(2) + '.csv'
    main(file_name)
time_end = tm.time()

print('DONE!!! time it took: ', (time_end-time1)/60, ' minutes')


INFO: Login to gaia TAP server [astroquery.gaia.core]
User: mharris
Password: ········
INFO: OK [astroquery.utils.tap.core]
INFO: Login to gaia data server [astroquery.gaia.core]
INFO: OK [astroquery.utils.tap.core]
500 Error 500:
null
500 Error 500:
null
500 Error 500:
null
500 Error 500:
null
500 Error 500:
null


KeyboardInterrupt: 

In [1]:

file_numbers = list(range(1, 14)) + list(range(27, 40))
# indx_num = 0
file_name = '../data/TIC_and_Gaia_ids_sector_s' + str(file_numbers[0]).zfill(2) + '.csv'


reader = pd.read_csv(
    file_name,
    names=["tic_id", "dr2_source_id"],
    dtype={"tic_id": "int64", "dr2_source_id": "int64"},
    index_col = 0, header= 0, 
    chunksize=100000
)


NameError: name 'pd' is not defined

In [2]:
import pandas as pd
import pyvo
from astropy.table import Table
from pathlib import Path
import time
import itertools

# --- Two mirror endpoints (no ESA front-end involved) ---
AIP = "https://gaia.aip.de/tap"                  # Gaia@AIP TAP mirror  [1](https://gaia.aip.de/metadata/gaiaedr3/dr2_neighbourhood/)
ARI = "https://gaia.ari.uni-heidelberg.de/tap"   # ARI/GAVO Gaia TAP    [2](https://gaia.ari.uni-heidelberg.de/)

# We'll try AIP first, then ARI if AIP throws 5xx at submit:
for tap_url in (AIP, ARI):
    try:
        tap = pyvo.dal.TAPService(tap_url)
        break
    except Exception as ex:
        print(f"Failed to init TAP service {tap_url}: {ex}")
else:
    raise RuntimeError("Could not initialize any TAP service")

# --- Input (keep both columns locally; upload only dr2_source_id) ---
file_numbers = list(range(1, 14)) + list(range(27, 40))
file_name = '../data/TIC_and_Gaia_ids_sector_s' + str(file_numbers[0]).zfill(2) + '.csv'


reader = pd.read_csv(
    file_name,
    names=["tic_id", "dr2_source_id"],
    dtype={"tic_id": "int64", "dr2_source_id": "int64"},
    index_col = 0, header= 0, 
    chunksize=25000
)
adql = """
SELECT n.dr2_source_id, n.dr3_source_id, n.angular_distance, n.magnitude_difference
FROM gaiadr3.dr2_neighbourhood AS n
JOIN TAP_UPLOAD.{tbl} AS u
  ON u.dr2_source_id = n.dr2_source_id
"""

outdir = Path("dr2_to_dr3_results")
outdir.mkdir(parents=True, exist_ok=True)

def run_chunk(df, chunk_idx, tap):
    """
    Submit one chunk, wait for completion using AsyncTAPJob.wait(),
    then fetch, reduce to one-best match, and persist.
    """
    from astropy.table import Table
    import pyvo
    import time

    # Upload as VOTable (reliable for TAP_UPLOAD)
    up = Table.from_pandas(df[["dr2_source_id"]])
    table_name = "dr2_ids_chunk"

    # We'll try primary, then fail over once if submit throws (e.g., 5xx)
    submit_attempts = [(tap, "primary")]
    AIP = "https://gaia.aip.de/tap"
    ARI = "https://gaia.ari.uni-heidelberg.de/tap"
    if tap.baseurl.rstrip("/") == AIP.rstrip("/"):
        submit_attempts.append((pyvo.dal.TAPService(ARI), "failover"))
    else:
        submit_attempts.append((pyvo.dal.TAPService(AIP), "failover"))

    last_err = None
    job = None
    for t, label in submit_attempts:
        try:
            job = t.submit_job(
                adql.format(tbl=table_name),
                uploads={table_name: up}
            )
            print(f"[chunk {chunk_idx:04d}] submitted on {label} ({t.baseurl})")
            break
        except Exception as ex:
            last_err = ex
            print(f"[chunk {chunk_idx:04d}] submit failed on {label} ({t.baseurl}): {ex}")
            time.sleep(2)

    if job is None:
        raise last_err

    # Start the job
    job.run()

    # ---- Poll using wait() (works across pyvo versions) ----
    try:
        # Wait up to ~4 minutes; adjust as you like
        job.wait(phases=["COMPLETED", "ERROR", "ABORTED"], timeout=240)
    except Exception as ex:
        # If wait() itself fails (rare), fall back to a small manual sleep and proceed
        print(f"[chunk {chunk_idx:04d}] wait() raised: {ex} -- continuing to check phase")

    # Check final phase
    phase = getattr(job, "phase", None)  # property that fetches state
    if phase not in ("COMPLETED",):
        # Try to surface server message if available
        err_txt = ""
        try:
            err_txt = job.error_summary.message.content
        except Exception:
            pass
        try:
            job.delete()
        except Exception:
            pass
        raise RuntimeError(f"Job not completed (phase={phase}). {err_txt or ''}")

    # Fetch results
    res = job.fetch_result().to_table().to_pandas()

    # One-best per DR2 (closest, then |ΔG|)
    if not res.empty:
        res.sort_values(["dr2_source_id", "angular_distance", "magnitude_difference"], inplace=True)
        res = res.groupby("dr2_source_id", as_index=False).first()

    # Persist per chunk
    res.to_parquet(outdir / f"dr2_to_dr3_chunk_{chunk_idx:04d}.parquet", index=False)

    # Clean up server-side job
    try:
        job.delete()
    except Exception:
        pass

    return len(res)
# Drive the chunks
total_rows = 0
for idx, df in enumerate(reader, start=1):
    try:
        n = run_chunk(df, idx, tap)
        total_rows += n
    except Exception as ex:
        print(f"[chunk {idx:04d}] ERROR: {ex}  -- Will continue with next chunk.")
        # Optional: add backoff or a requeue mechanism here
4.905, 
print(f"Done. Wrote {len(list(outdir.glob('*.parquet')))} chunk files; {total_rows} rows of matches.")

[chunk 0001] submitted on primary (https://gaia.aip.de/tap)
[chunk 0002] submitted on primary (https://gaia.aip.de/tap)
[chunk 0003] submitted on primary (https://gaia.aip.de/tap)
[chunk 0004] submitted on primary (https://gaia.aip.de/tap)
[chunk 0005] submitted on primary (https://gaia.aip.de/tap)
[chunk 0006] submitted on primary (https://gaia.aip.de/tap)
[chunk 0007] submitted on primary (https://gaia.aip.de/tap)
[chunk 0008] submitted on primary (https://gaia.aip.de/tap)
[chunk 0009] submitted on primary (https://gaia.aip.de/tap)
[chunk 0010] submitted on primary (https://gaia.aip.de/tap)
[chunk 0011] submitted on primary (https://gaia.aip.de/tap)
[chunk 0012] submitted on primary (https://gaia.aip.de/tap)
Done. Wrote 12 chunk files; 281097 rows of matches.


In [9]:
import numpy as np
per = np.array([4.905, 29.344, 1.635 ])

In [11]:
per/min(per)

array([ 3.        , 17.94740061,  1.        ])

In [12]:
29.344/4.905

5.982466870540265